In [1]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

cwd = os.getcwd()
if cwd.endswith('notebook'):
    os.chdir('..')
    cwd = os.getcwd()

In [2]:
sns.set_palette('colorblind')
sns.set_style('whitegrid')
sns.set_context('paper', font_scale=1.8)
plt.rcParams['font.family'] = 'Helvetica'

palette = sns.color_palette().as_hex()

data_folder = Path('./data')
assert data_folder.is_dir()

figures_folder = Path('./figures')
assert figures_folder.is_dir()

In [27]:
gtdb_metadata = pd.read_csv(data_folder / 'gtdb_metadata.csv', index_col='ncbi_accession')
archaeal_accessions = set(gtdb_metadata[gtdb_metadata['domain'] == 'Archaea'].index)

In [28]:
archaeal_hits = pd.read_csv(data_folder / 'pg_synthesis' / 'pg_synthesis_archaeal_hits.csv')
print(f'Number of archaeal hits: {len(archaeal_hits):,}')
archaeal_hits.head()

Number of archaeal hits: 42,070


,target_name,accession,query_name,accession_query,full_evalue,full_score,full_bias,dom_evalue,dom_score,dom_bias,exp,reg,clu,ov,env,dom,rep,inc
0,MCI5866507.1@GCA_022768985.1,GCA_022768985.1,newDdlB,-,9.100000e-92,320.2,0.3,1.000000e-91,320.0,0.3,1.0,1,0,0,1,1,1,1
1,WTHQ01000015.1_4@GCA_011523055.1,GCA_011523055.1,newDdlB,-,1.100000e-81,287.1,0.1,1.200000e-81,286.9,0.1,1.0,1,0,0,1,1,1,1
2,PIN78533.1@GCA_002762865.1,GCA_002762865.1,newDdlB,-,5.200000e-74,261.8,0.0,6.500000e-74,261.5,0.0,1.0,1,0,0,1,1,1,1
3,JALRLN010000114.1_3@GCA_023254445.1,GCA_023254445.1,newDdlB,-,2.300000e-67,240.0,0.0,2.800000e-67,239.7,0.0,1.0,1,0,0,1,1,1,1
4,NQU98779.1@GCA_013202845.1,GCA_013202845.1,newDdlB,-,3.900000e-65,232.7,0.7,4.800000e-65,232.4,0.7,1.0,1,0,0,1,1,1,1


In [36]:
top_hits = archaeal_hits[
    archaeal_hits['full_evalue'] <= 1e-6
][
    ['accession', 'target_name', 'query_name', 'full_score']
].sort_values(
    ['accession', 'query_name', 'full_score'],
    ascending=[True, True, False],
).drop_duplicates(
    ['accession', 'query_name']
)
print(f'Number of hits: {len(top_hits):,}')
top_hits.head()

Number of hits: 15,429


,accession,target_name,query_name,full_score
16619,GCA_000008085.1,AAR38988.1@GCA_000008085.1,newFtsZ,380.2
844,GCA_000016605.1,ABP96130.1@GCA_000016605.1,newDdlB,86.9
25262,GCA_000016605.1,ABP96273.1@GCA_000016605.1,newMraY,100.1
28650,GCA_000016605.1,ABP96011.1@GCA_000016605.1,newMurA,45.3
28808,GCA_000016605.1,ABP94902.1@GCA_000016605.1,newMurB,45.2


In [37]:
grouped_df = top_hits[['query_name', 'accession']].groupby('query_name').nunique().sort_values('accession', ascending=False)
grouped_df['percent'] = (100 * grouped_df['accession'] / len(archaeal_accessions)).round(1)
grouped_df

,accession,percent
query_name,,
newFtsZ,3260,88.0
newDdlB,2670,72.0
newMraY,1902,51.3
newMurA,1792,48.4
newMurE,1341,36.2
newMurF,1328,35.8
newMurG,1141,30.8
newMurC,915,24.7
newMurD,619,16.7


In [58]:
top_hits_with_phylum = pd.merge(
    top_hits,
    gtdb_metadata[
        gtdb_metadata['domain'] == 'Archaea'
    ].reset_index()[
        ['ncbi_accession', 'gtdb_phylum', 'gtdb_class']
    ].rename(columns={'ncbi_accession': 'accession'}),
    on='accession',
    how='left',
)
top_hits_with_phylum.head()

,accession,target_name,query_name,full_score,gtdb_phylum,gtdb_class
0,GCA_000008085.1,AAR38988.1@GCA_000008085.1,newFtsZ,380.2,Nanoarchaeota,Nanoarchaeia
1,GCA_000016605.1,ABP96130.1@GCA_000016605.1,newDdlB,86.9,Thermoproteota,Thermoprotei_A
2,GCA_000016605.1,ABP96273.1@GCA_000016605.1,newMraY,100.1,Thermoproteota,Thermoprotei_A
3,GCA_000016605.1,ABP96011.1@GCA_000016605.1,newMurA,45.3,Thermoproteota,Thermoprotei_A
4,GCA_000016605.1,ABP94902.1@GCA_000016605.1,newMurB,45.2,Thermoproteota,Thermoprotei_A


In [59]:
top_hits_with_phylum[
    ['gtdb_phylum', 'query_name', 'accession']
].groupby(['gtdb_phylum', 'accession']).nunique().reset_index()[
    ['gtdb_phylum', 'query_name']
].groupby('gtdb_phylum').mean().sort_values('query_name', ascending=False)

,query_name
gtdb_phylum,
Methanobacteriota,8.953333
Altiarchaeota,5.961538
Iainarchaeota,5.588235
Halobacteriota,5.182679
JACRDV01,5.000000
Thermoplasmatota,4.396985
Methanobacteriota_B,4.368421
Hydrothermarchaeota,4.312500
Asgardarchaeota,3.888298


In [ ]:
class_g = top_hits_with_phylum[
    ['gtdb_class', 'query_name', 'accession']
].groupby(['gtdb_class', 'accession']).nunique().reset_index()[
    ['gtdb_class', 'query_name']
].groupby('gtdb_class').mean().sort_values('query_name', ascending=False)

class_g.loc[['Methanobacteria', 'Methanopyri', 'Methanococci']]

,query_name
gtdb_class,
Methanobacteria,9.622807
Methanopyri,9.666667
Methanococci,6.575758


In [65]:
top_hits_with_phylum[
    ['gtdb_class', 'accession']
].groupby(['gtdb_class']).nunique().loc[
    ['Methanobacteria', 'Methanopyri', 'Methanococci', 'Halobacteria']
]

,accession
gtdb_class,
Methanobacteria,114
Methanopyri,3
Methanococci,33
Halobacteria,385


In [62]:
class_g2 = top_hits_with_phylum[
    ['gtdb_class', 'query_name', 'accession']
].groupby(['gtdb_class', 'query_name']).nunique()

class_g2.loc[['Methanobacteria', 'Methanopyri', 'Methanococci', 'Halobacteria']]

accession
gtdb_class      query_name           
Methanobacteria newDdlB           114
                newFtsA            98
                newFtsI             2
                newFtsW             2
                newFtsZ           105
                newMraW             1
                newMraY           113
                newMurA           105
                newMurC           114
                newMurD           114
                newMurE           114
                newMurF           114
                newMurG           101
Methanopyri     newDdlB             3
                newFtsA             3
                newFtsZ             3
                newMraY             2
                newMurA             3
                newMurC             3
                newMurD             3
                newMurE             3
                newMurF             3
                newMurG             3
Methanococci    newDdlB            33
                newFtsZ            33
                newMraY            31
                newMurA            32
                newMurC             8
                newMurD            17
                newMurE             2
                newMurF            30
                newMurG            31
Halobacteria    newDdlB           376
                newFtsA            20
                newFtsI             2
                newFtsW             1
                newFtsZ           385
                newMraY             1
                newMurA           364
                newMurB            10
                newMurC           200
                newMurD            10
                newMurE           374
                newMurF           367
                newMurG            26

## Load PGH proteins

In [82]:
pgh_df = pd.read_csv(data_folder / 'pgh_proteins.csv')
archaeal_pgh = pgh_df[pgh_df['domain'] == 'Archaea'].set_index('gtdb_class')
archaeal_pgh.head()

,assembly_accession,domain,gtdb_phylum,gtdb_order,gtdb_family,gtdb_genus,gtdb_species,ncbi_organism_name,protein_id,pgh_architecture
gtdb_class,,,,,,,,,,
Aenigmatarchaeia,GCA_003663345.1,Archaea,Aenigmatarchaeota,PWEA01,B50-G16,B50-G16,B50-G16 sp003663345,Candidatus Aenigmarchaeota archaeon,RLJ01288.1,PG_binding_3+Glyco_hydro_108
Aenigmatarchaeia,GCA_018304545.1,Archaea,Aenigmatarchaeota,CG10238-14,CG10238-14,JAGVVZ01,JAGVVZ01 sp018304545,Candidatus Aenigmarchaeota archaeon,MBS3052716.1,LysM+Amidase_2
Aenigmatarchaeia,GCA_025058005.1,Archaea,Aenigmatarchaeota,CG10238-14,CG10238-14,JAHLMN01,JAHLMN01 sp025058005,Candidatus Aenigmarchaeota archaeon,MCS7135447.1,LysM+NLPC_P60
Aenigmatarchaeia,GCA_015661515.1,Archaea,Aenigmatarchaeota,Aenigmatarchaeales,SZUA-1535,SZUA-1535,SZUA-1535 sp015661515,Nanoarchaeota archaeon,HIQ49991.1,PG_binding_3+Glyco_hydro_108
Altiarchaeia,GCA_902384455.1,Archaea,Altiarchaeota,IMC4,SCGC-AAA252-I15,CABMCB01,CABMCB01 sp902384455,uncultured archaeon,VVB52780.1,PG_binding_3+Glyco_hydro_108


In [75]:
n_assemblies_per_class = gtdb_metadata[
    gtdb_metadata['domain'] == 'Archaea'
][['gtdb_class', 'accession']].groupby(['gtdb_class']).nunique().rename(columns={
    'accession': 'n_accessions'
})

n_pgh_per_class = archaeal_pgh.reset_index()[
    ['gtdb_class', 'assembly_accession']
].groupby(['gtdb_class']).nunique().rename(columns={
    'assembly_accession': 'n_accessions_with_pgh'
})

stats_per_class = pd.merge(
    n_assemblies_per_class,
    n_pgh_per_class,
    how='left',
    on='gtdb_class',
).fillna(0.0)

stats_per_class['percent_pgh'] = (100 * stats_per_class['n_accessions_with_pgh'] / stats_per_class['n_accessions']).round(2)

stats_per_class.loc[['Methanobacteria', 'Methanopyri', 'Methanococci', 'Halobacteria']]

,n_accessions,n_accessions_with_pgh,percent_pgh
gtdb_class,,,
Methanobacteria,114,2.0,1.75
Methanopyri,3,0.0,0.00
Methanococci,33,0.0,0.00
Halobacteria,385,44.0,11.43


## Putting everything together

"Among the proteins encoded by dcw cluster genes, the four muramyl ligase enzymes (MurC, MurD, MurE, and MurF), and the D-alanine–D-alanine ligase (Ddl) are critical for PG biosynthesis." (Lupo et al., 2025)

Let's focus only on assemblies with all 5 hits.

In [77]:
essential_hits = ['newDdlB', 'newMurC', 'newMurD', 'newMurE', 'newMurF']

v = top_hits_with_phylum[
    top_hits_with_phylum['query_name'].isin(essential_hits)
][['accession', 'query_name']].groupby('accession').nunique()

accession_with_all_essential_hits = sorted(set(v[v['query_name'] == len(essential_hits)].index))

In [95]:
len(accession_with_all_essential_hits)

353

In [80]:
gtdb_metadata.loc[
    accession_with_all_essential_hits][['gtdb_class', 'accession']
].groupby('gtdb_class').nunique().sort_values('accession', ascending=False).rename(columns={
    'accession': 'n_with_partial_cluster'
})

,accession
gtdb_class,
Methanobacteria,114
Thermoplasmata,47
Methanosarcinia,34
Nanoarchaeia,33
Micrarchaeia,21
Iainarchaeia,17
Bathyarchaeia,16
E2,11
Thermococci,9


In [90]:
n_with_essential = len(archaeal_pgh[archaeal_pgh['assembly_accession'].isin(accession_with_all_essential_hits)])
p = 100 * n_with_essential / len(accession_with_all_essential_hits)

print(f'N with essential PG synthesis homologs = {n_with_essential:,} of {len(accession_with_all_essential_hits)} ({p:.0f} %)')

N with essential PG synthesis homologs = 21 of 353 (6 %)


In [92]:
n_all_without_essentials = len(archaeal_accessions) - len(accession_with_all_essential_hits)

n_without_essential = len(archaeal_pgh[~archaeal_pgh['assembly_accession'].isin(accession_with_all_essential_hits)])
p = 100 * n_without_essential / n_all_without_essentials

print(f'N without essential PG synthesis homologs = {n_without_essential:,} of {n_all_without_essentials:,} ({p:.0f} %)')

N without essential PG synthesis homologs = 189 of 3,353 (6 %)


In [96]:
gtdb_metadata[gtdb_metadata['gtdb_genus'] == 'Halogranum']

,accession,ambiguous_bases,checkm_completeness,checkm_contamination,checkm_marker_count,checkm_marker_lineage,checkm_marker_set_count,checkm_strain_heterogeneity,coding_bases,coding_density,...,trna_aa_count,trna_count,trna_selenocysteine_count,domain,gtdb_phylum,gtdb_class,gtdb_order,gtdb_family,gtdb_genus,gtdb_species
ncbi_accession,,,,,,,,,,,,,,,,,,,,,
GCF_000283335.1,RS_GCF_000283335.1,0,98.61,0.76,417,f__Halobacteriaceae (UID96),263,0.0,3886827,86.521866,...,19,67,0,Archaea,Halobacteriota,Halobacteria,Halobacteriales,Haloferacaceae,Halogranum,Halogranum rubrum
GCF_900103715.1,RS_GCF_900103715.1,0,99.57,0.76,417,f__Halobacteriaceae (UID96),263,0.0,3302031,87.582685,...,19,53,0,Archaea,Halobacteriota,Halobacteria,Halobacteriales,Haloferacaceae,Halogranum,Halogranum gelatinilyticum
GCF_900110465.1,RS_GCF_900110465.1,0,99.38,1.77,417,f__Halobacteriaceae (UID96),263,0.0,4347805,83.842362,...,19,59,0,Archaea,Halobacteriota,Halobacteria,Halobacteriales,Haloferacaceae,Halogranum,Halogranum amylolyticum


In [97]:
'GCF_000283335.1' in accession_with_all_essential_hits

False

In [103]:
top_hits_with_phylum[top_hits_with_phylum['accession'] == 'GCF_000283335.1']

,accession,target_name,query_name,full_score,gtdb_phylum,gtdb_class
12708,GCF_000283335.1,WP_009375483.1@GCF_000283335.1,newDdlB,74.3,Halobacteriota,Halobacteria
12709,GCF_000283335.1,WP_009366313.1@GCF_000283335.1,newFtsZ,408.3,Halobacteriota,Halobacteria
12710,GCF_000283335.1,WP_009374601.1@GCF_000283335.1,newMurA,73.3,Halobacteriota,Halobacteria
12711,GCF_000283335.1,WP_009366113.1@GCF_000283335.1,newMurC,52.2,Halobacteriota,Halobacteria
12712,GCF_000283335.1,WP_009366113.1@GCF_000283335.1,newMurE,81.9,Halobacteriota,Halobacteria
12713,GCF_000283335.1,WP_009366113.1@GCF_000283335.1,newMurF,74.7,Halobacteriota,Halobacteria
